<a href="https://colab.research.google.com/github/vdlucas-queiroz/CadImobi/blob/main/COTIA_prompt_condominio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# RECONSTRUÇÃO TABELA DE LANÇAMENTO DE CONDOMÍNIOS
# Vinicius D'Lucas Bezerra e Queiroz

In [ ]:
!pip install adjustText
!pip install simplekml
!pip install folium pyproj
!pip cache purge
!pip install -U google-genai
!pip install pymupdf

Files removed: 18


In [ ]:
import pandas as pd
import numpy as np
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
from adjustText import adjust_text
import folium
from pyproj import Transformer

from google.colab import files
from google import genai
from google.colab import userdata
import re
import time
import json
import fitz


In [ ]:
# 1. Configuração da API
try:
    api_key = userdata.get('GEMINI_API_KEY_2')
    client = genai.Client(api_key=api_key)
    print("API Configurada com sucesso!")
except Exception as e:
    print("ERRO: Configure o Secret 'GEMINI_API_KEY' no menu lateral.")

def obter_tabela_via_ai(caminho_arquivo, prompt):
    """
    Envia o arquivo (PDF ou Imagem) diretamente para o Gemini.
    A IA faz o OCR (leitura de imagem) e a extração dos dados em um único passo.
    """

    # Busca o nome correto do modelo disponívelA
    nome_modelo = 'gemini-2.5-flash' # Versão estável e rápida

    # 2. Upload do arquivo para a API do Google (suporta PDF e Imagens)
    print(f"Fazendo upload do arquivo: {caminho_arquivo}...")
    sample_file = client.files.upload(file=caminho_arquivo)

    # Aguarda o processamento do arquivo pelo Google
    while sample_file.state.name == "PROCESSING":
        print(".", end="")
        time.sleep(2)
        sample_file = client.files.get(name=sample_file.name)

    # 3. Prompt especializado (ajustado para ler páginas 3 a 7) #AJUSTAR O PROMPT

    print("Gemini analisando o documento (OCR + Extração)...")
    response = client.models.generate_content(
        model=nome_modelo,
        contents=[sample_file, prompt],
        config={
            "response_mime_type": "application/json"
        }
    )

  # 4. Limpeza (opcional, mas boa prática)
    client.files.delete(name=sample_file.name)

    return response.text

def obter_tabela_via_ai_2(caminho_arquivo, prompt):
    nome_modelo = 'gemini-2.5-flash'

    print(f"Fazendo upload do arquivo: {caminho_arquivo}...")
    sample_file = client.files.upload(file=caminho_arquivo)

    # Aguarda o processamento
    while sample_file.state.name == "PROCESSING":
        time.sleep(30)
        sample_file = client.files.get(name=sample_file.name)

    print("Gemini analisando o documento...")

    # MELHORIA: Adicionando Schema ou System Instruction para forçar a leitura de todas as páginas
    response = client.models.generate_content(
        model=nome_modelo,
        contents=[
            sample_file,
            "Analise cada página individualmente. Não pule nenhuma matrícula. " + prompt
        ],
        config={
            "response_mime_type": "application/json",
            # Temperatura 0 garante respostas mais fiéis ao texto e menos "criativas"
            "temperature": 0.0
        }
    )

    client.files.delete(name=sample_file.name)
    return response.text


API Configurada com sucesso!


In [ ]:
prompt = """
# Contexto
Você é um extrator de dados de alta precisão para registros de imóveis.
Analise o PDF e extraia os dados técnicos da matrícula. Cada página do arquivo contém uma matrícula.


# Campos para Extrair:
- MATRICULA
- AREA_TERRENO
- AREA_OCUPADA
- TESTADA_1 (tamanho da frente do imóvel)
- AREA_EDIFICADA (área construída)
- FI (Fração Ideal)

# Formato de Saída (JSON Estrito):
Retorne uma LISTA de objetos JSON. Não use Markdown, não explique.
Exemplo: [{"MATRICULA": 123, "AREA_TERRENO": 500.00, ...}]

# Se o dado não existir:
Use null (sem aspas). Nunca invente dados"""

In [ ]:
prompt = """# Contexto
Você é um especialista em conferência cartorial. Sua tarefa é extrair dados técnicos de matrículas de imóveis com 100% de precisão.

# Regras de Ouro para Evitar Confusão:
0. Cada página possui uma matrícula... a quantidade de linhas é a quantidade de páginas
1. IDENTIFICAÇÃO: Cada matrícula começa com o termo "Matrícula".
2. ISOLAMENTO: Todos os dados (Área, Testada, FI) extraídos DEVEM pertencer estritamente ao bloco de texto daquela matrícula específica. Nunca misture dados de blocos diferentes.
3. VALIDAÇÃO: Se houver mais de uma área mencionada (ex: área do terreno e área da construção), atribua corretamente conforme o mapeamento abaixo.


# Campos para Extração:
- MATRICULA: Apenas os dígitos numéricos.
- AREA_TERRENO: Procure por "área de...", "terreno com...", "medindo...".
- AREA_OCUPADA: Projeção horizontal da benfeitoria no solo (se disponível).
- TESTADA_1: A medida da "frente", "face" ou "divisa com a rua/logradouro".
- AREA_EDIFICADA: Área construída averbada (Ignore áreas privativas se for apartamento, foque na área total da unidade).
- FI: Fração Ideal (formato decimal).

# Formato de Saída (JSON Estrito):
Retorne uma LISTA de objetos JSON. Não use Markdown, não explique.
Exemplo: [{"MATRICULA": 123, "AREA_TERRENO": 500.00, ...}]

# Se o dado não existir:
Use null (sem aspas). Nunca invente dados."""

In [ ]:
# --- CONFIGURAÇÃO ---
arquivo_path = '/content/condominio_jd_das_rosas.pdf'
tamanho_lote = 20
lista_final_dfs = []

# Abrimos o PDF para contar as páginas e fatiar
doc = fitz.open(arquivo_path)
total_paginas = len(doc)

print(f"Iniciando processamento de {total_paginas} páginas em lotes de {tamanho_lote}...")

# --- LOOP POR LOTES ---
for i in range(0, total_paginas, tamanho_lote):
    inicio = i
    fim = min(i + tamanho_lote, total_paginas)

    print(f"Extraindo lote: Páginas {inicio + 1} a {fim}...")

    # 1. Criamos um PDF temporário na memória com apenas o lote atual
    lote_pdf = fitz.open()
    lote_pdf.insert_pdf(doc, from_page=inicio, to_page=fim-1)
    pdf_bytes = lote_pdf.write()

    # Salvamos temporariamente para a função de IA ler (ou ajuste para ler bytes)
    temp_lote_nome = f"lote_atual.pdf"
    with open(temp_lote_nome, "wb") as f:
        f.write(pdf_bytes)

    # 2. Ajuste do Prompt para o Lote Atual
    prompt_lote = f"""Analise as páginas deste documento (Páginas {inicio+1} a {fim}).
    Extraia cada matrícula como um objeto individual.
    Regra estrita: Se uma matrícula começar nesta página e terminar na próxima, extraia apenas os dados contidos neste bloco.

    {prompt}""" # Reaproveita o seu prompt de regras e campos

    try:
        # 3. Chamada da IA para o lote específico
        resultado_bruto = obter_tabela_via_ai_2(temp_lote_nome, prompt_lote)

        # Limpeza e Conversão
        codigo_limpo = re.sub(r'```.*?```|```', '', resultado_bruto, flags=re.DOTALL).strip()
        dados_json = json.loads(codigo_limpo)

        if dados_json:
            df_lote = pd.DataFrame(dados_json)

            # Padronização de nomes de colunas do JSON (lowercase para evitar erro)
            df_lote.columns = [c.lower() for c in df_lote.columns]
            lista_final_dfs.append(df_lote)

    except Exception as e:
        print(f"Erro no lote {inicio+1}-{fim}: {e}")

    lote_pdf.close()

# --- CONSOLIDAÇÃO FINAL ---
if lista_final_dfs:
    df_final = pd.concat(lista_final_dfs, ignore_index=True)

    print(f"\n--- SUCESSO TOTAL ---")
    print(f"Total de Matrículas consolidadas: {len(df_final)}")
    display(df_final.dropna(axis=1, how='all'))
else:
    print("Nenhum dado foi extraído com sucesso.")

doc.close()

Iniciando processamento de 190 páginas em lotes de 20...
Extraindo lote: Páginas 1 a 20...
Fazendo upload do arquivo: lote_atual.pdf...
Gemini analisando o documento...
Extraindo lote: Páginas 21 a 40...
Fazendo upload do arquivo: lote_atual.pdf...
Gemini analisando o documento...
Extraindo lote: Páginas 41 a 60...
Fazendo upload do arquivo: lote_atual.pdf...
Gemini analisando o documento...
Extraindo lote: Páginas 61 a 80...
Fazendo upload do arquivo: lote_atual.pdf...
Gemini analisando o documento...
Extraindo lote: Páginas 81 a 100...
Fazendo upload do arquivo: lote_atual.pdf...
Gemini analisando o documento...
Extraindo lote: Páginas 101 a 120...
Fazendo upload do arquivo: lote_atual.pdf...
Gemini analisando o documento...
Extraindo lote: Páginas 121 a 140...
Fazendo upload do arquivo: lote_atual.pdf...
Gemini analisando o documento...
Extraindo lote: Páginas 141 a 160...
Fazendo upload do arquivo: lote_atual.pdf...
Gemini analisando o documento...
Extraindo lote: Páginas 161 a 180

/tmp/ipython-input-3171470882.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(lista_final_dfs, ignore_index=True)


,matricula,area_terreno,area_edificada,fi
0,152853,NaN,120.385,0.006248
1,152854,NaN,120.247,0.006245
2,152855,NaN,113.085,0.005972
3,152856,NaN,119.409,0.006217
4,152857,NaN,120.031,0.006239
...,...,...,...,...
183,153038,NaN,83.504,0.005066
184,153039,NaN,83.504,0.005066
185,153040,NaN,82.296,0.004980
186,153041,NaN,82.296,0.004980


In [ ]:
df_0 = pd.read_csv('/content/planilha.txt')

In [ ]:
df_0.columns

Index(['FOLHA ;PARAMETRO ;QUADRA ;LOTE ;BLOCO ;ECONOMIA ;FACE ;INSCRIÇÃO ANTIGA ;MATRÍCULA ;CÓDIGO DE PESSOA;PROPRIETÁRIO ;CPF ;RG ;ORGAO EXP. ;CNPJ ;CO-PROPRIETÁRIO;CPF;RG;ORGAO EXP.;CNPJ ;PROCESSO;ALVARA;PROCESSO DE INDIVIDUALIZAÇÃO ;LOCAL DO IMÓVEL ;Nº ;Complemento;Telefone;NOME DO BAIRRO: ;CEP ;CIDADE ;UF;ENDEREÇO PARA ENTREGA ;Nº ;COMPLEMENTO ;TELEFONE: ();NOME DO BAIRRO: ;CEP ;CIDADE ;UF;AREA DO TERRENO ;ÁREA OCUPADA ;TESTADA 1;TESTADA 2;TESTADA 3;TESTADA 4;Nº TESTADAS;QTD DE EDIFICAÇÕES ;CÓD. DE COBRANÇA ;ESTADO DA EDIFICAÇÃO ;FORMA ;SITUAÇÃO ;USO ;BENFEITORIAS ;CATEGORIA PROPRIEDADE;ÁREA EDIFICADA;Nº VAGAS DESCOBERTAS ;Nº VAGAS COBERTAS ;ANO CONSTRUÇÃO ;Nº PAVIMENTOS ;PISCINA ;OCUPAÇÃO ;REGIME DE OCUPAÇÃO ;CATEGORIA DE USO ;SERVIÇOS MUNICIPAIS ;CASA ALINHADA ;CASA RECUADA ;APTO ;ESCRITORIO;OUTROS ;ESTADO DE CONSERVAÇÃO ;REVEST. EXTERNO ;REVEST. INTERNO ;PINTURA EXTERNA ;PINTURA INTERNA ;PISO ;FORRO ;INSTALAÇÃO ELETRICA ;INSTALAÇÃO SANITÁRIA ;ESTRUTURA ;COBERTURA;ESQUADRIAS ;ELE

In [ ]:
df = df_final.copy()

In [ ]:
df_final

,matricula,area_terreno,area_ocupada,testada_1,area_edificada,fi
0,152853,NaN,None,None,120.385,0.006248
1,152854,NaN,None,None,120.247,0.006245
2,152855,NaN,None,None,113.085,0.005972
3,152856,NaN,None,None,119.409,0.006217
4,152857,NaN,None,None,120.031,0.006239
...,...,...,...,...,...,...
183,153038,NaN,None,None,83.504,0.005066
184,153039,NaN,None,None,83.504,0.005066
185,153040,NaN,None,None,82.296,0.004980
186,153041,NaN,None,None,82.296,0.004980


In [ ]:
df_final.columns

Index(['matricula', 'area_terreno', 'area_ocupada', 'testada_1',
       'area_edificada', 'fi'],
      dtype='object')

In [ ]:
area_do_terreno = 7868.84
area_construida = 11758.86

In [ ]:
usar_fi = True
if usar_fi:
    df['area_terreno'] = df['fi']*area_do_terreno
    print(df['area_terreno'] )

if usar_fi:

    df['area_edificada'] = df['fi']*area_construida
    print(df['area_edificada'])

0      49.164512
1      49.140906
2      46.992712
3      48.920578
4      49.093693
         ...    
183    39.863543
184    39.863543
185    39.186823
186    39.186823
187    40.453706
Name: area_terreno, Length: 188, dtype: float64
0      73.469357
1      73.434081
2      70.223912
3      73.104833
4      73.363528
         ...    
183    59.570385
184    59.570385
185    58.559123
186    58.559123
187    60.452299
Name: area_edificada, Length: 188, dtype: float64


In [ ]:
df['area_ocupada'] = None

In [ ]:
df

,matricula,area_terreno,area_ocupada,testada_1,area_edificada,fi
0,152853,49.164512,None,None,73.469357,0.006248
1,152854,49.140906,None,None,73.434081,0.006245
2,152855,46.992712,None,None,70.223912,0.005972
3,152856,48.920578,None,None,73.104833,0.006217
4,152857,49.093693,None,None,73.363528,0.006239
...,...,...,...,...,...,...
183,153038,39.863543,None,None,59.570385,0.005066
184,153039,39.863543,None,None,59.570385,0.005066
185,153040,39.186823,None,None,58.559123,0.004980
186,153041,39.186823,None,None,58.559123,0.004980


In [ ]:
new_record_data_1 = {
    'matricula': [153018],
    'fi': [0.005141],
    'area_terreno': [0.005141 * area_do_terreno],
    'area_edificada': [0.005141 * area_construida],
    'area_ocupada': [None],
    'testada_1': [None]
}

new_record_data_2 = {
    'matricula': [152994],
    'fi': [0.005141],
    'area_terreno': [0.005141 * area_do_terreno],
    'area_edificada': [0.005141 * area_construida],
    'area_ocupada': [None],
    'testada_1': [None]
}

new_df_1 = pd.DataFrame(new_record_data_1)
new_df_2 = pd.DataFrame(new_record_data_2)

df = pd.concat([df, new_df_1, new_df_2], ignore_index=True)
df = df.sort_values(by='matricula').reset_index(drop=True)

print("DataFrame atualizado com os novos registros e ordenado:")
display(df)

DataFrame atualizado com os novos registros e ordenado:


,matricula,area_terreno,area_ocupada,testada_1,area_edificada,fi
0,152853,49.164512,None,None,73.469357,0.006248
1,152854,49.140906,None,None,73.434081,0.006245
2,152855,46.992712,None,None,70.223912,0.005972
3,152856,48.920578,None,None,73.104833,0.006217
4,152857,49.093693,None,None,73.363528,0.006239
...,...,...,...,...,...,...
185,153038,39.863543,None,None,59.570385,0.005066
186,153039,39.863543,None,None,59.570385,0.005066
187,153040,39.186823,None,None,58.559123,0.004980
188,153041,39.186823,None,None,58.559123,0.004980


In [ ]:
df[df['matricula']==152944]

,matricula,area_terreno,area_ocupada,testada_1,area_edificada,fi
91,152944,39.186823,None,None,58.559123,0.00498


In [ ]:
file_name = "jardim_das_rosas_IA.csv"
df.to_csv(file_name, index=False, decimal= ',', sep=';', encoding='utf-8-sig')
files.download(file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Rode isso uma vez para ver os nomes exatos disponíveis na sua conta
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/embedding-001
models/text-embedding-004
models/gemini-embedding-001
models/aqa
models/